# Composer — Experiment Orchestration Layer

This notebook provides a comprehensive guide to MalthusJAX's Composer, the high-level orchestration layer for defining, running, and comparing evolutionary algorithms.

**What You'll Learn:**
- Three entry points: quick_run(), from_toml(), compare()
- String DSL for operators (e.g., "tournament:num_selections=25,tournament_size=3")
- Reproducible experiments via TOML configuration
- Multi-pipeline fair comparison
- Result aggregation and analysis
- Data management and custom configurations
- Backend flexibility (MalthusJAX + Evosax)

## Part 1: Composer Architecture Overview

**Composer** abstracts boilerplate configuration and provides three entry points:

```
┌─────────────────────────────────────────────┐
│  Composer (quick_run, from_toml, compare)   │
├─────────────────────────────────────────────┤
│  Config Parser:                             │
│    - TOML → dictionaries                     │
│    - String specs → operator instances       │
├─────────────────────────────────────────────┤
│  Catalogs & Registries:                      │
│    - GenomeCatalog ("real:dim=10")           │
│    - OperatorCatalog ("tournament:...")      │
├─────────────────────────────────────────────┤
│  Engine Factories:                          │
│    - MalthusJAX (native JAX)                │
│    - Evosax (alternative backend)           │
├─────────────────────────────────────────────┤
│  BenchmarkRunner:                           │
│    - Run across seeds                       │
│    - Collect results & history              │
├─────────────────────────────────────────────┤
│  Results:                                   │
│    - ExperimentResult (multi-seed)          │
│    - ComparisonResult (multi-pipeline)      │
└─────────────────────────────────────────────┘
```

### Three Entry Points

| Entry Point | Use Case | Input | Output |
|-------------|----------|-------|--------|
| `quick_run()` | Interactive exploration | Python kwargs + string specs | ExperimentResult |
| `from_toml()` | Reproducible experiments | TOML file | ExperimentResult or ComparisonResult |
| `compare()` | Fair algorithm comparison | Dict of pipelines | ComparisonResult |

### Result Types

| Type | Scope | Key Methods |
|------|-------|-------------|
| **ExperimentResult** | Single experiment, multiple seeds | `.aggregated_summary()`, `.combined_history()` |
| **ComparisonResult** | Multiple pipelines, each multi-seed | `.summary_table()`, `.plot_convergence()` |
| **RunResult** | Single seeded run | `.to_dict()`, `.from_dict()` |

## Part 2: String DSL — Declarative Operator Specifications

Composer uses a declarative string format to specify operators without code:

```
"operator_name:param1=val1,param2=val2,param3=val3"
```

### Fitness Functions

| Spec | Example | Description |
|------|---------|-------------|
| `sphere` | `"sphere:dim=10"` | Minimize $\sum x_i^2$ |
| `rastrigin` | `"rastrigin:dim=10"` | Multimodal benchmark |
| `griewank` | `"griewank:dim=10"` | Multimodal with peaks |
| `bbob` | `"bbob:fn=3,dims=10"` | Black-Box Optimization suite |
| `tsp` | `"tsp:data_id=berlin52"` | Traveling Salesman Problem |
| `knapsack` | `"knapsack:capacity=500,num_items=50"` | 0/1 knapsack |
| `binary_sum` | `"binary_sum:length=20"` | Sum of bits |

### Selection Operators

| Spec | Example | Characteristics |
|------|---------|------------------|
| `tournament` | `"tournament:num_selections=25,tournament_size=3"` | Competitive, tunable pressure |
| `roulette` | `"roulette:num_selections=30"` | Fitness-proportional, smooth |
| `elite_pool` | `"elite_pool:num_selections=25,elite_k=10"` | High exploitation |

### Crossover Operators (Real-Valued)

| Spec | Example | Type |
|------|---------|------|
| `uniform_real` | `"uniform_real"` | Per-gene exchange |
| `blend` | `"blend:alpha=0.5"` | Blended interpolation |
| `simulated_binary` | `"simulated_binary:eta=20"` | SBX (polynomial kernel) |
| `binomial` | `"binomial:cr=0.7"` | DE-style crossover |

### Crossover Operators (Binary)

| Spec | Example | Type |
|------|---------|------|
| `uniform_binary` | `"uniform_binary"` | Bit-wise mixing |
| `single_point` | `"single_point"` | Single-point cut |

### Mutation Operators (Real-Valued)

| Spec | Example | Type |
|------|---------|------|
| `gaussian` | `"gaussian:mutation_rate=0.1,mutation_strength=0.1"` | Normal noise |
| `ball` | `"ball:mutation_rate=0.1"` | Uniform hypersphere |
| `polynomial` | `"polynomial:mutation_rate=0.1,eta=20"` | Polynomial kernel |

### Mutation Operators (Binary)

| Spec | Example | Type |
|------|---------|------|
| `bitflip` | `"bitflip:mutation_rate=0.05"` | Independent bit flips |
| `scramble` | `"scramble"` | Random reordering |
| `swap` | `"swap"` | Pairwise swaps |

## Part 3: Quick-Start with quick_run()

For interactive exploration, use `quick_run()` with string specs and keyword arguments.

### Basic Pattern

```python
from malthusjax.composer import Composer

composer = Composer.create_default()

result = composer.quick_run(
    fitness="sphere:dim=10",
    selection="tournament:num_selections=25,tournament_size=3",
    crossover="blend:alpha=0.5",
    mutation="gaussian:mutation_rate=0.1,mutation_strength=0.1",
    pop_size=50,
    generations=100,
    seeds=(42, 43, 44),
)

# Analyze results
summary = result.aggregated_summary()
print(f"Best: {summary['best_fitness']['mean']:.4f} ± {summary['best_fitness']['stdev']:.4f}")
```

### Result Access

```python
# Single-seed metrics
first_run = result.runs[0]
print(f"Seed {first_run.seed}: best={first_run.metrics['best_fitness']:.4f}")

# Multi-seed aggregation
agg = result.aggregated_summary()
print(f"Mean best: {agg['best_fitness']['mean']:.4f}")
print(f"Std best:  {agg['best_fitness']['stdev']:.4f}")

# Convergence history (flattened)
history = result.combined_history(seed_field="seed")
import pandas as pd
df = pd.DataFrame(history)
print(df.head())
```

## Part 4: Reproducible Experiments with TOML

For production and sharing, define experiments in TOML files:

### TOML Structure

```toml
[experiment]
name = "my_experiment"
output_dir = "results/my_experiment"

[experiment.shared]
# Defaults for all pipelines
fitness = "sphere:dim=10"
selection = "tournament:num_selections=25,tournament_size=3"
pop_size = 50
generations = 100
seeds = [42, 43, 44]
bounds = [-5.0, 5.0]
maximize = false

[pipelines.blend_gaussian]
crossover = "blend:alpha=0.5"
mutation = "gaussian:mutation_rate=0.5,mutation_strength=0.1"

[pipelines.sbx_polynomial]
crossover = "simulated_binary:eta=20"
mutation = "polynomial:mutation_rate=0.1,eta=20"
```

### Executing from TOML

```python
from malthusjax.composer import Composer

# Execute all pipelines
result = Composer.from_toml("experiment.toml")

# Or execute specific pipelines
result = Composer.from_toml("experiment.toml", pipelines=["blend_gaussian"])

# Access results (same as quick_run)
print(result.summary_table())
result.plot_convergence(seed_index=0)
```

## Part 5: Multi-Pipeline Fair Comparison

Use `compare()` to benchmark multiple algorithms with fair initialization:

### Basic Pattern

```python
comparison = composer.compare(
    pipelines={
        "Blend+Gaussian": {
            "crossover": "blend:alpha=0.5",
            "mutation": "gaussian:mutation_rate=0.1,mutation_strength=0.1",
        },
        "SBX+Polynomial": {
            "crossover": "simulated_binary:eta=20",
            "mutation": "polynomial:mutation_rate=0.1,eta=20",
        },
    },
    fitness="sphere:dim=10",  # shared
    selection="tournament:num_selections=25,tournament_size=3",  # shared
    pop_size=50,
    generations=100,
    seeds=(42, 43, 44),
    shared_initial_population=True,  # Fair: same starting point
)

# Compare results
table = comparison.summary_table()
for pipeline_name, metrics in table.items():
    print(f"{pipeline_name}: best={metrics['best_fitness']:.4f}")

# Plot convergence curves
comparison.plot_convergence(seed_index=0)
```

### ComparisonResult Methods

| Method | Returns | Use Case |
|--------|---------|----------|
| `.summary_table()` | Per-pipeline metrics (aggregated) | Quick comparison |
| `.plot_convergence(seed_index=0)` | Matplotlib figure | Visualize curves |
| `.convergence_data(seed_index=0)` | Raw history dict | Custom plotting |
| `.pipelines` | Dict of ExperimentResult | Detailed per-pipeline analysis |

## Part 6: Data Management & Advanced Patterns

### Mechanism 1: Programmatic Data (quick_run + data_config)

```python
# Interactive TSP with synthetic distance matrix
result = composer.quick_run(
    fitness="tsp:data_id=problem_50",
    data_config={
        "problem_50": {
            "source": "synthetic",
            "num_cities": 50,
            "random_seed": 42,
        },
    },
    selection="tournament:num_selections=25,tournament_size=3",
    mutation="swap:mutation_rate=0.1",
    pop_size=100,
    generations=200,
    seeds=(42, 43, 44),
    genome_type="categorical",
)

summary = result.aggregated_summary()
print(f"Best tour length: {summary['best_fitness']['mean']:.2f}")
```

### Mechanism 2: Declarative Data (TOML [data.*] sections)

```toml
[experiment.shared]
fitness = "tsp:data_id=berlin52"

[data.berlin52]
source = "file"
path = "data/tsp/berlin52.tsp"

[pipelines.ga_swap]
mutation = "swap:mutation_rate=0.1"
```

Then:
```python
result = Composer.from_toml("experiment.toml")
```

### Data Sources

| Source | Configuration | Use Case |
|--------|---------------|----------|
| `synthetic` | Generate on-the-fly | Benchmark functions, random TSP |
| `file` | Load from disk | TSPLIB instances, knapsack data |

**Example: Synthetic Knapsack**
```toml
[data.items_50]
source = "synthetic"
num_items = 50
capacity = 500
weight_range = [10, 50]
value_range = [30, 150]
random_seed = 42
```

## Part 7: Backend Flexibility

Composer supports multiple evolutionary backends:

### MalthusJAX Backend (Default)

```python
result = composer.quick_run(
    backend="malthusjax",
    engine_type="ga",
    fitness="sphere:dim=10",
    selection="tournament:num_selections=25,tournament_size=3",
    crossover="blend:alpha=0.5",
    mutation="gaussian:mutation_rate=0.1,mutation_strength=0.1",
    elitism=2,  # MalthusJAX-specific
    pop_size=50,
    generations=100,
)
```

**Features:**
- Full operator customization
- Mutation scheduling
- Static resource budgeting
- Multi-device support

### Evosax Backend (Alternative)

```python
result = composer.quick_run(
    backend="evosax",
    evosax_strategy="DifferentialEvolution",  # Strategy name
    fitness="sphere:dim=10",
    pop_size=30,
    generations=100,
    seeds=(42, 43, 44),
)
```

**Available Strategies:**
- SimpleGA — Basic genetic algorithm
- MR15_GA — Mutation-rate adaptive GA
- DifferentialEvolution — DE algorithm
- And 20+ more (see Evosax docs)

### Mixed Comparison

```python
comparison = composer.compare(
    pipelines={
        "MalthusJAX GA": {
            "backend": "malthusjax",
            "crossover": "blend:alpha=0.5",
            "mutation": "gaussian:mutation_rate=0.1",
        },
        "Evosax DE": {
            "backend": "evosax",
            "evosax_strategy": "DifferentialEvolution",
        },
    },
    fitness="sphere:dim=10",
    pop_size=50,
    generations=100,
)

table = comparison.summary_table()
```

## Part 8: Result Analysis & Export

### ExperimentResult Methods

```python
result = composer.quick_run(...)

# Multi-seed aggregation
agg = result.aggregated_summary()
# Returns: {"best_fitness": {"mean": X, "stdev": Y, "median": Z}, ...}

# Flattened history for pandas
history = result.combined_history(seed_field="seed")
import pandas as pd
df = pd.DataFrame(history)
df.to_csv("convergence.csv")

# Serialization
result_dict = result.to_dict()
import json
with open("result.json", "w") as f:
    json.dump(result_dict, f)
```

### ComparisonResult Methods

```python
comparison = composer.compare(...)

# Summary statistics per pipeline
table = comparison.summary_table()
for name, metrics in table.items():
    print(f"{name}: {metrics['best_fitness']:.4f}")

# Convergence visualization
comparison.plot_convergence(seed_index=0)

# Raw data for custom plotting
data = comparison.convergence_data(seed_index=0)
# Returns: {"Pipeline A": [{"generation": 0, "best_fitness": X}, ...], ...}
```

## Part 9: Developer Checklist

### Quick-Start Workflow

- [ ] Define search space: `genome_type`, `genome_length`, `bounds`
- [ ] Choose fitness: `fitness="sphere:dim=10"` or custom spec
- [ ] Select selection: `tournament`, `roulette`, or `elite_pool`
- [ ] Choose crossover: `blend`, `simulated_binary`, etc. (or skip for no recombination)
- [ ] Choose mutation: `gaussian`, `polynomial`, `bitflip`, etc.
- [ ] Set population size: `pop_size` (powers of 2 preferred for GPU)
- [ ] Run: `composer.quick_run(...)`
- [ ] Analyze: `.aggregated_summary()`, `.combined_history()`

### Reproducible Experiment Workflow

- [ ] Write TOML with `[experiment.shared]` + `[pipelines.*]`
- [ ] Include `[data.*]` sections if using custom data
- [ ] Test with single pipeline: `from_toml(path, pipelines=["test"])`
- [ ] Run full comparison: `from_toml(path)`
- [ ] Export results: `.to_dict()`, `.combined_history()`
- [ ] Version control TOML file for reproducibility

### Comparison Workflow

- [ ] Define baseline algorithm
- [ ] Define alternative algorithms (operator variations)
- [ ] Use `compare()` with `shared_initial_population=True`
- [ ] Call `.summary_table()` for per-algorithm metrics
- [ ] Call `.plot_convergence()` for per-seed visualization
- [ ] Specify seeds consistently across all pipelines

### Troubleshooting

| Error | Cause | Fix |
|-------|-------|-----|
| `KeyError: "sphere_2"` | Operator spec typo | Check spelling: `"sphere:dim=10"` (no underscore) |
| `TypeError: expected float, got str` | TOML type mismatch | Unquote numbers: `dim=10` not `dim="10"` |
| `FileNotFoundError` | TOML file not found | Use absolute path or verify relative to cwd |
| `ValueError: Invalid operator` | Unknown operator name | Check OperatorCatalog for valid names |

In [1]:
# Setup: Import required libraries

import jax
import pandas as pd

print("JAX version:", jax.__version__)
print("Available devices:", jax.devices())

# Note: In actual use, you would import from malthusjax.composer
# For this notebook, we'll simulate key Composer API patterns

jax.config.update("jax_enable_x64", True)

JAX version: 0.10.0
Available devices: [CpuDevice(id=0)]


In [2]:
# Example 1: Simulating quick_run() API
"""
Demonstrates Composer.quick_run() API patterns.
Shows string DSL, parameter passing, and result access.
"""

print("Composer API Example: quick_run()\n")
print("="*70)
print()
print("Usage pattern:")
print("""
from malthusjax.composer import Composer

composer = Composer.create_default()

result = composer.quick_run(
    # Objective function (string DSL)
    fitness="sphere:dim=10",
    
    # Genetic operators (string DSL)
    selection="tournament:num_selections=25,tournament_size=3",
    crossover="blend:alpha=0.5",
    mutation="gaussian:mutation_rate=0.1,mutation_strength=0.1",
    
    # Population & generations
    pop_size=50,
    generations=100,
    seeds=(42, 43, 44),
    
    # Search space
    genome_type="real",
    genome_length=10,
    bounds=(-5.0, 5.0),
    maximize=False,  # minimize by default
)

# Analyze results
print("Accessing results:")
print()
print("  # Single-seed run")
print("  run = result.runs[0]")
print(f"  run.seed = 42")
print(f"  run.metrics['best_fitness'] = 0.123")
print()
print("  # Multi-seed aggregation")
print("  agg = result.aggregated_summary()")
print(f"  agg['best_fitness'] = {{'mean': 0.123, 'stdev': 0.045, 'median': 0.110}}")
print()
print("  # Export to pandas")
print("  history = result.combined_history(seed_field='seed')")
print("  df = pd.DataFrame(history)")
print("  df.to_csv('convergence.csv')")
print()
print("="*70)
print()
print("✓ Composer abstracts boilerplate → focus on operator specs")
""")

Composer API Example: quick_run()


Usage pattern:

from malthusjax.composer import Composer

composer = Composer.create_default()

result = composer.quick_run(
    # Objective function (string DSL)
    fitness="sphere:dim=10",

    # Genetic operators (string DSL)
    selection="tournament:num_selections=25,tournament_size=3",
    crossover="blend:alpha=0.5",
    mutation="gaussian:mutation_rate=0.1,mutation_strength=0.1",

    # Population & generations
    pop_size=50,
    generations=100,
    seeds=(42, 43, 44),

    # Search space
    genome_type="real",
    genome_length=10,
    bounds=(-5.0, 5.0),
    maximize=False,  # minimize by default
)

# Analyze results
print("Accessing results:")
print()
print("  # Single-seed run")
print("  run = result.runs[0]")
print(f"  run.seed = 42")
print(f"  run.metrics['best_fitness'] = 0.123")
print()
print("  # Multi-seed aggregation")
print("  agg = result.aggregated_summary()")
print(f"  agg['best_fitness'] = {{'mean': 0.123, 'stdev': 0.045,

In [3]:
# Example 2: String DSL Catalog Reference
"""
Displays complete catalog of available operators and their specs.
"""

# Create comprehensive catalog reference
catalog = {
    'Fitness Functions': {
        'sphere': 'sphere:dim=10',
        'rastrigin': 'rastrigin:dim=10',
        'griewank': 'griewank:dim=10',
        'bbob': 'bbob:fn=3,dims=10',
        'tsp': 'tsp:data_id=berlin52',
        'knapsack': 'knapsack:capacity=500,num_items=50',
    },
    'Selection Operators': {
        'tournament': 'tournament:num_selections=25,tournament_size=3',
        'roulette': 'roulette:num_selections=30',
        'elite_pool': 'elite_pool:num_selections=25,elite_k=10',
    },
    'Crossover (Real)': {
        'uniform': 'uniform_real',
        'blend': 'blend:alpha=0.5',
        'sbx': 'simulated_binary:eta=20',
        'binomial': 'binomial:cr=0.7',
    },
    'Crossover (Binary)': {
        'uniform': 'uniform_binary',
        'single_point': 'single_point',
    },
    'Mutation (Real)': {
        'gaussian': 'gaussian:mutation_rate=0.1,mutation_strength=0.1',
        'ball': 'ball:mutation_rate=0.1',
        'polynomial': 'polynomial:mutation_rate=0.1,eta=20',
    },
    'Mutation (Binary)': {
        'bitflip': 'bitflip:mutation_rate=0.05',
        'scramble': 'scramble',
        'swap': 'swap',
    },
}

print("Composer String DSL Catalog\n")
for category, specs in catalog.items():
    print(f"\n{category}:")
    print("─" * 60)
    for name, spec in specs.items():
        print(f"  {name:20} → {spec}")

print("\n" + "="*60)
print("\n✓ Use any combination to define evolutionary algorithms")

Composer String DSL Catalog


Fitness Functions:
────────────────────────────────────────────────────────────
  sphere               → sphere:dim=10
  rastrigin            → rastrigin:dim=10
  griewank             → griewank:dim=10
  bbob                 → bbob:fn=3,dims=10
  tsp                  → tsp:data_id=berlin52
  knapsack             → knapsack:capacity=500,num_items=50

Selection Operators:
────────────────────────────────────────────────────────────
  tournament           → tournament:num_selections=25,tournament_size=3
  roulette             → roulette:num_selections=30
  elite_pool           → elite_pool:num_selections=25,elite_k=10

Crossover (Real):
────────────────────────────────────────────────────────────
  uniform              → uniform_real
  blend                → blend:alpha=0.5
  sbx                  → simulated_binary:eta=20
  binomial             → binomial:cr=0.7

Crossover (Binary):
────────────────────────────────────────────────────────────
  uniform       

In [4]:
# Example 3: TOML Configuration Examples
"""
Demonstrates TOML structure for reproducible experiments.
"""

# Example 1: Sphere optimization
toml_example_1 = """[experiment]
name = "sphere_comparison"
output_dir = "results/sphere_bench"

[experiment.shared]
fitness = "sphere:dim=25"
selection = "tournament:num_selections=30,tournament_size=3"
pop_size = 100
generations = 200
seeds = [42, 43, 44]
bounds = [-5.0, 5.0]
maximize = false

[pipelines.uniform_gaussian]
crossover = "uniform_real"
mutation = "gaussian:mutation_rate=0.5,mutation_strength=0.1"

[pipelines.blend_gaussian]
crossover = "blend:alpha=0.5"
mutation = "gaussian:mutation_rate=0.5,mutation_strength=0.1"

[pipelines.sbx_polynomial]
crossover = "simulated_binary:eta=20"
mutation = "polynomial:mutation_rate=0.1,eta=20"
"""

# Example 2: TSP with data
toml_example_2 = """[experiment]
name = "tsp_benchmark"
output_dir = "results/tsp"

[experiment.shared]
genome_type = "categorical"
pop_size = 100
generations = 300
seeds = [42, 43, 44]
elitism = 2

[data.berlin52]
source = "file"
path = "data/tsp/berlin52.tsp"

[data.synthethic_50]
source = "synthetic"
num_cities = 50
random_seed = 123

[pipelines.on_berlin]
fitness = "tsp:data_id=berlin52"
selection = "tournament:num_selections=50,tournament_size=3"
mutation = "swap:mutation_rate=0.1"

[pipelines.on_synthetic_50]
fitness = "tsp:data_id=synthethic_50"
selection = "tournament:num_selections=50,tournament_size=3"
mutation = "swap:mutation_rate=0.1"
"""

# Example 3: Knapsack
toml_example_3 = """[experiment]
name = "knapsack_0_1"
output_dir = "results/knapsack"

[experiment.shared]
genome_type = "binary"
pop_size = 80
generations = 150
seeds = [1, 2, 3]

[data.items_50]
source = "synthetic"
num_items = 50
capacity = 500
weight_range = [10, 50]
value_range = [30, 150]
random_seed = 42

[pipelines.uniform_bitflip]
fitness = "knapsack:data_id=items_50"
selection = "tournament:num_selections=40,tournament_size=3"
crossover = "uniform_binary"
mutation = "bitflip:mutation_rate=0.05"

[pipelines.single_point_bitflip]
fitness = "knapsack:data_id=items_50"
selection = "roulette:num_selections=40"
crossover = "single_point"
mutation = "bitflip:mutation_rate=0.05"
"""

print("Composer TOML Examples\n")
print("="*70)
print()
print("Example 1: Sphere Optimization Comparison\n")
print(toml_example_1)
print()
print("="*70)
print()
print("Example 2: TSP Benchmarking on Different Instances\n")
print(toml_example_2)
print()
print("="*70)
print()
print("Example 3: 0/1 Knapsack Problem\n")
print(toml_example_3)

print()
print("="*70)
print("\n✓ Version control TOML for reproducibility")
print("✓ Easy team collaboration and paper sharing")

Composer TOML Examples


Example 1: Sphere Optimization Comparison

[experiment]
name = "sphere_comparison"
output_dir = "results/sphere_bench"

[experiment.shared]
fitness = "sphere:dim=25"
selection = "tournament:num_selections=30,tournament_size=3"
pop_size = 100
generations = 200
seeds = [42, 43, 44]
bounds = [-5.0, 5.0]
maximize = false

[pipelines.uniform_gaussian]
crossover = "uniform_real"
mutation = "gaussian:mutation_rate=0.5,mutation_strength=0.1"

[pipelines.blend_gaussian]
crossover = "blend:alpha=0.5"
mutation = "gaussian:mutation_rate=0.5,mutation_strength=0.1"

[pipelines.sbx_polynomial]
crossover = "simulated_binary:eta=20"
mutation = "polynomial:mutation_rate=0.1,eta=20"



Example 2: TSP Benchmarking on Different Instances

[experiment]
name = "tsp_benchmark"
output_dir = "results/tsp"

[experiment.shared]
genome_type = "categorical"
pop_size = 100
generations = 300
seeds = [42, 43, 44]
elitism = 2

[data.berlin52]
source = "file"
path = "data/tsp/berlin52.tsp"

[dat

In [5]:
# Example 4: Results Analysis & Visualization Patterns
"""
Demonstrates result aggregation, export, and visualization.
"""

# Simulate results structure (in practice, from composer.quick_run/Composer.from_toml)
print("Results Analysis Patterns\n")
print("="*70)

# Simulated aggregated summary
agg_summary = {
    'best_fitness': {
        'mean': 0.1234,
        'stdev': 0.0456,
        'median': 0.1100,
        'min': 0.0890,
        'max': 0.1890,
    },
    'mean_fitness': {
        'mean': 2.3450,
        'stdev': 0.1200,
        'median': 2.3100,
    },
    'duration_seconds': {
        'mean': 15.32,
        'stdev': 0.45,
    },
}

print("\n1. ExperimentResult.aggregated_summary()\n")
for metric, stats in agg_summary.items():
    print(f"  {metric}:")
    for stat_name, stat_val in stats.items():
        print(f"    {stat_name:8} = {stat_val:.4f}")

print()
print("="*70)
print()
print("2. ExperimentResult.combined_history()\n")

# Simulated history
history_sample = [
    {'generation': 0, 'best_fitness': 10.50, 'mean_fitness': 15.32, 'seed': 42},
    {'generation': 1, 'best_fitness': 9.23, 'mean_fitness': 13.45, 'seed': 42},
    {'generation': 2, 'best_fitness': 7.89, 'mean_fitness': 11.23, 'seed': 42},
    {'generation': 0, 'best_fitness': 10.80, 'mean_fitness': 15.67, 'seed': 43},
]

df = pd.DataFrame(history_sample)
print("  Flattened history (ready for pandas/CSV):")
print()
print(df.to_string(index=False))
print()
print("  Export: df.to_csv('convergence.csv')")

print()
print("="*70)
print()
print("3. ComparisonResult.summary_table()\n")

comparison_table = {
    "Blend+Gaussian": {
        'best_fitness': 0.0890,
        'mean_fitness': 2.1200,
        'duration_seconds': 15.32,
    },
    "SBX+Polynomial": {
        'best_fitness': 0.0650,
        'mean_fitness': 1.9800,
        'duration_seconds': 16.78,
    },
}

for pipeline_name, metrics in comparison_table.items():
    print(f"  {pipeline_name}:")
    for metric_name, value in metrics.items():
        print(f"    {metric_name:20} = {value:.4f}")
    print()

print("="*70)
print()
print("✓ Results easily exported to pandas, JSON, CSV")
print("✓ Convergence histories enable detailed analysis")

Results Analysis Patterns


1. ExperimentResult.aggregated_summary()

  best_fitness:
    mean     = 0.1234
    stdev    = 0.0456
    median   = 0.1100
    min      = 0.0890
    max      = 0.1890
  mean_fitness:
    mean     = 2.3450
    stdev    = 0.1200
    median   = 2.3100
  duration_seconds:
    mean     = 15.3200
    stdev    = 0.4500


2. ExperimentResult.combined_history()

  Flattened history (ready for pandas/CSV):

 generation  best_fitness  mean_fitness  seed
          0         10.50         15.32    42
          1          9.23         13.45    42
          2          7.89         11.23    42
          0         10.80         15.67    43

  Export: df.to_csv('convergence.csv')


3. ComparisonResult.summary_table()

  Blend+Gaussian:
    best_fitness         = 0.0890
    mean_fitness         = 2.1200
    duration_seconds     = 15.3200

  SBX+Polynomial:
    best_fitness         = 0.0650
    mean_fitness         = 1.9800
    duration_seconds     = 16.7800


✓ Results easily

## Summary

**Key Takeaways:**

1. **Three Entry Points**: 
   - `quick_run()` — Interactive exploration with string specs
   - `from_toml()` — Reproducible experiments with version control
   - `compare()` — Fair algorithm comparison

2. **String DSL**: Powerful, declarative operator specifications without code
   - Format: `"operator_name:param1=val1,param2=val2"`
   - Covers genomes, fitness, selection, crossover, mutation

3. **TOML for Reproducibility**:
   - `[experiment.shared]` for defaults
   - `[pipelines.*]` for variants
   - `[data.*]` for custom/synthetic data sources

4. **Fair Comparison**:
   - Use `compare()` with `shared_initial_population=True`
   - Ensures differences are algorithmic, not initialization variance

5. **Result Analysis**:
   - `.aggregated_summary()` for multi-seed statistics
   - `.combined_history()` for pandas/CSV export
   - `.plot_convergence()` for visualization

6. **Backend Flexibility**:
   - MalthusJAX (native JAX, full control)
   - Evosax (alternative strategies, different dynamics)
   - Mixed comparisons across backends

**Best Practices:**
- Start with `quick_run()` for exploration
- Move to TOML once configuration is stable
- Use `compare()` for production algorithm comparisons
- Version control TOML configurations
- Export results to CSV for external analysis
- Specify seeds consistently for reproducibility